# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [ ]:
import pandas as pd
from IPython.display import display

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
original = pd.read_csv(url)
df = original.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
df = df.loc[:, ~df.columns.str.startswith("unnamed")].dropna(how="all").copy()
for column in df.select_dtypes(include="object"):
    df[column] = df[column].astype("string").str.strip()
df["state"] = df["state"].replace({"AZ": "Arizona", "WA": "Washington", "Cali": "California"})
df["gender"] = df["gender"].str.upper().replace({"FEMALE": "F", "FEMAL": "F", "MALE": "M"})
df["education"] = df["education"].replace({"Bachelors": "Bachelor"})
df["response"] = df["response"].str.title()
for column in ["total_claim_amount", "monthly_premium_auto", "customer_lifetime_value", "number_of_open_complaints", "number_of_policies"]:
    df[column] = pd.to_numeric(df[column], errors="raise")
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y", errors="raise")
df["month"] = df["effective_to_date"].dt.to_period("M").astype("string")
print("Duplicados exactos eliminados:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Nulos por columna:")
print(df.isna().sum())
# No inventamos respuesta ni importes; los resumenes ignoran valores ausentes.
# Se conservan filas distintas del mismo cliente: no sabemos si son otras polizas.
display(df.head())

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [ ]:
low_claim_responders = df.loc[(df["total_claim_amount"] < 1000) & df["response"].eq("Yes")].copy()
display(low_claim_responders)
print("Registros seleccionados:", len(low_claim_responders))
assert low_claim_responders["total_claim_amount"].lt(1000).all()
assert low_claim_responders["response"].eq("Yes").all()

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [ ]:
responders = df.loc[df["response"].eq("Yes")]
segment_summary = responders.groupby(["policy_type", "gender"], dropna=False).agg(records=("customer", "size"), customers=("customer", "nunique"), mean_monthly_premium=("monthly_premium_auto", "mean"), mean_clv=("customer_lifetime_value", "mean"), mean_claim=("total_claim_amount", "mean"), median_claim=("total_claim_amount", "median"), observed_claims=("total_claim_amount", "count"))
display(segment_summary.round(2))
print("Segmentos con menor siniestro medio:")
display(segment_summary.sort_values("mean_claim").head(3).round(2))
print("Segmentos con mayor CLV medio:")
display(segment_summary.sort_values("mean_clv", ascending=False).head(3).round(2))
print("Un siniestro menor y CLV mayor son indicadores descriptivos. No prueban beneficio: faltan costes, exposicion y un periodo comun para primas y siniestros. Revisar el tamano de cada segmento.")

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [ ]:
customers_by_state = df.groupby("state")["customer"].nunique().rename("customers").sort_values(ascending=False)
states_over_500 = customers_by_state.loc[customers_by_state > 500].to_frame()
display(customers_by_state.to_frame())
display(states_over_500)
assert states_over_500["customers"].gt(500).all()

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [ ]:
clv_stats = df.groupby(["education", "gender"], dropna=False)["customer_lifetime_value"].agg(["count", "min", "max", "median"])
display(clv_stats.round(2))
print("Mayor mediana (educacion, genero):", clv_stats["median"].idxmax())
print("El maximo puede estar condicionado por casos extremos; la mediana describe mejor el cliente central. Las diferencias entre grupos no establecen causalidad ni justifican decisiones individuales.")

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [ ]:
# Segun la pista, contamos registros como aproximacion de polizas.
# number_of_policies es cartera del cliente, no ventas mensuales verificadas.
policies_by_state_month = df.groupby(["state", "month"]).size().unstack("month", fill_value=0)
display(policies_by_state_month)
print("El mes procede de effective_to_date, no de una fecha de venta; esta tabla no acredita ventas reales.")

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [ ]:
top3_states = policies_by_state_month.sum(axis=1).sort_values(ascending=False).head(3).index
top3_policies = policies_by_state_month.loc[top3_states].reset_index().melt(id_vars="state", var_name="month", value_name="policy_records").sort_values(["state", "month"]).reset_index(drop=True)
display(top3_policies)
print("Top 3 por total de todos los meses:", top3_states.tolist())

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [ ]:
known_response = df.loc[df["response"].isin(["Yes", "No"])].copy()
known_response["responded"] = known_response["response"].eq("Yes").astype(int)
channel_summary = known_response.groupby("sales_channel").agg(records=("responded", "size"), yes_responses=("responded", "sum"), response_rate=("responded", "mean"))
channel_summary["response_rate_pct"] = channel_summary["response_rate"] * 100
display(channel_summary.sort_values("response_rate", ascending=False).round(2))
# melt convierte la tabla ancha de tasas en una tabla larga.
rate_wide = channel_summary[["response_rate_pct"]].T
rate_wide.index.name = "metric"
response_rates_long = rate_wide.reset_index().melt(id_vars="metric", var_name="sales_channel", value_name="response_rate_pct")
display(response_rates_long.round(2))
assert channel_summary["response_rate"].between(0, 1).all()
print("Mejor tasa observada:", channel_summary["response_rate"].idxmax())
print("Sales channel es una aproximacion al canal de marketing. Asociacion no equivale a efecto causal; faltan asignacion aleatoria y costes. La tasa se calcula por registro con respuesta conocida.")

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

Solucion con asistencia de IA. Las conclusiones concretas se generan con los datos al ejecutar el cuaderno; no se han inventado resultados. Se distingue cliente unico de registro y cartera de polizas de ventas reales. Los nulos de respuesta no entran en el denominador de tasas.